# Cold Chain Forecasting — Air Temperature & Humidity (LSTM)

Predict the next **30 minutes** of `t_air_c` and `rh_pct` from the previous **120 minutes** of sensor readings.

**Critical structural note:** this dataset is **200 independent trips**, not one continuous time series.
Every interpolation, rolling op, and sliding window must be done **inside `groupby("trip_id")`**.
A window that spans the end of one trip and the start of the next describes a physical transition
that never happened.

The `split` column is already assigned **per trip** (each of the 200 trips belongs entirely to one
split), so it is leak-free. Use it as-is — do not re-split.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Window configuration
N_PAST   = 120   # minutes of history fed to the model
N_FUTURE = 30    # minutes to forecast

# Shortest trip in the data is 241 rows, so N_PAST + N_FUTURE must stay under 241
assert N_PAST + N_FUTURE <= 241, "Window too long - some trips would be dropped entirely"

In [ ]:
CSV_PATH = "data/coldchain_sensors.csv"   # adjust if your path differs

coldChainDF = pd.read_csv(CSV_PATH)
print(coldChainDF.shape)
coldChainDF.head()

In [ ]:
# Sanity checks on the structure we are relying on
print("trips:", coldChainDF.trip_id.nunique())
print("trips per split:")
print(coldChainDF.groupby("split").trip_id.nunique())

# Each trip must live in exactly one split (this is what makes the split leak-free)
assert (coldChainDF.groupby("trip_id").split.nunique() == 1).all()

# Sampling must be exactly 1 minute with no gaps inside a trip
diffs = coldChainDF.groupby("trip_id")["minute"].diff().dropna().unique()
print("minute diffs:", diffs)
assert set(diffs) == {1.0}

print("\nmissing values:")
print(coldChainDF.isna().sum()[lambda s: s > 0])

## 2. Handle missing values (per trip)

`t_air_c` and `rh_pct` are the only columns with gaps, and they always go missing **together**
(sensor dropout, not independent noise). The gaps are 344 values across 148 trips, in runs of
only 1–2 minutes — short enough that linear interpolation is safe.

The `groupby` is what keeps the interpolation from bleeding the tail of one trip into the head
of the next.

In [ ]:
TARGETS = ["t_air_c", "rh_pct"]

# Inspect gap run lengths before deciding to interpolate
flag = coldChainDF.t_air_c.isna()
runs = (flag != flag.shift()).cumsum()[flag].value_counts().value_counts().sort_index()
print("gap run lengths (length -> count):")
print(runs)

coldChainDF[TARGETS] = (
    coldChainDF.groupby("trip_id")[TARGETS]
    .transform(lambda s: s.interpolate(limit_direction="both"))
)

assert coldChainDF[TARGETS].isna().sum().sum() == 0
print("\nno missing values remaining")

## 3. Cyclical encoding of `hour_of_day`

`hour_of_day` runs 0–24 and wraps around 76 times in this dataset. Fed in raw, the model would
treat 23.9 and 0.1 as maximally far apart when they are one minute apart. Sine/cosine encoding
puts them next to each other on a circle.

In [ ]:
coldChainDF["hod_sin"] = np.sin(2 * np.pi * coldChainDF.hour_of_day / 24)
coldChainDF["hod_cos"] = np.cos(2 * np.pi * coldChainDF.hour_of_day / 24)

coldChainDF[["hour_of_day", "hod_sin", "hod_cos"]].head()

## 4. Feature groups

Three kinds of column, treated differently at scaling time:

- **Continuous** — standardized.
- **Binary flags** — left alone. Standardizing 0/1 columns just adds noise and makes them
  harder to interpret.
- **Cyclical** — already in [-1, 1], leave as-is.

This is the **past-only** formulation: the model sees the last `N_PAST` minutes of *everything*
and predicts the next `N_FUTURE` minutes of temp and humidity. If your real deployment knows the
compressor/door schedule in advance you can feed future exogenous values too, but get this
version working first.

In [ ]:
CONT_FEATS = ["t_air_c", "rh_pct", "t_ambient_c", "rh_ambient_pct"]
BIN_FEATS  = ["door_open", "vehicle_moving", "compressor_on", "defrost_on"]
CYC_FEATS  = ["hod_sin", "hod_cos"]

FEATURES = CONT_FEATS + BIN_FEATS + CYC_FEATS
N_FEATURES = len(FEATURES)

print(f"{N_FEATURES} input features:", FEATURES)
print("targets:", TARGETS)

## 5. Scaling — fit on train rows only

Two separate scalers:

- `feat_scaler` for the continuous inputs.
- `targ_scaler` for the two targets alone, so predictions can be inverted back to °C and %RH
  without slicing columns out of a wider scaler.

Both are fit on **train rows only**, then applied to all three splits. Fitting on the full frame
is the single most common leak in this kind of pipeline.

In [ ]:
train_mask = coldChainDF.split == "train"

feat_scaler = StandardScaler().fit(coldChainDF.loc[train_mask, CONT_FEATS])
targ_scaler = StandardScaler().fit(coldChainDF.loc[train_mask, TARGETS])

scaled = coldChainDF.copy()
scaled[CONT_FEATS] = feat_scaler.transform(coldChainDF[CONT_FEATS])

# Scaled copies of the targets, kept in separate columns so the originals stay readable
Y_COLS = ["y_t_air", "y_rh"]
scaled[Y_COLS] = targ_scaler.transform(coldChainDF[TARGETS])

scaled[CONT_FEATS + Y_COLS].describe().T[["mean", "std", "min", "max"]]

## 6. Build windows inside each trip

`make_windows` loops over trips within a single split and never crosses a trip boundary.

It also returns `last_obs` — the final observed target value in each input window. That is the
persistence baseline prediction, and carrying it alongside the windows saves recomputing it later.

In [ ]:
def make_windows(df, split, n_past=N_PAST, n_future=N_FUTURE):
    """Slide windows within each trip of `split`. Never crosses trip boundaries."""
    X, y, last_obs = [], [], []
    subset = df[df.split == split]

    for _, g in subset.groupby("trip_id", sort=False):
        feats = g[FEATURES].values
        targs = g[Y_COLS].values
        n_windows = len(g) - n_past - n_future + 1
        if n_windows <= 0:
            continue  # trip too short for this window size
        for i in range(n_windows):
            X.append(feats[i : i + n_past])
            y.append(targs[i + n_past : i + n_past + n_future])
            last_obs.append(targs[i + n_past - 1])

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        np.array(last_obs, dtype=np.float32),
    )


X_train, y_train, last_train = make_windows(scaled, "train")
X_val,   y_val,   last_val   = make_windows(scaled, "val")
X_test,  y_test,  last_test  = make_windows(scaled, "test")

for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s} X {X.shape}  y {y.shape}")

## 7. Persistence baseline — establish this before training

"The next 30 minutes look exactly like the last observed minute."

Cold chain air temperature is heavily autocorrelated, so this baseline is genuinely strong.
It is the number the LSTM has to beat to have learned anything worth having.

In [ ]:
def inverse_targets(arr):
    """(samples, n_future, 2) in scaled space -> original units."""
    flat = arr.reshape(-1, 2)
    return targ_scaler.inverse_transform(flat).reshape(arr.shape)


def report(y_true_scaled, y_pred_scaled, label):
    y_true = inverse_targets(y_true_scaled)
    y_pred = inverse_targets(y_pred_scaled)
    err = np.abs(y_true - y_pred)
    rmse = np.sqrt(((y_true - y_pred) ** 2).reshape(-1, 2).mean(axis=0))
    mae = err.reshape(-1, 2).mean(axis=0)
    print(f"{label}")
    print(f"  t_air_c  MAE {mae[0]:.4f} C     RMSE {rmse[0]:.4f}")
    print(f"  rh_pct   MAE {mae[1]:.4f} %RH   RMSE {rmse[1]:.4f}")
    return mae


# Repeat the last observed value across the whole forecast horizon
persist_test = np.repeat(last_test[:, None, :], N_FUTURE, axis=1)
baseline_mae = report(y_test, persist_test, "PERSISTENCE BASELINE (test)")

## 8. Model

In [ ]:
model = Sequential([
    Input(shape=(N_PAST, N_FEATURES)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(N_FUTURE * len(TARGETS)),
    Reshape((N_FUTURE, len(TARGETS))),
])

model.compile(
    optimizer=Adam(1e-3),
    loss="huber",      # less twitchy than MSE around door-open / defrost spikes
    metrics=["mae"],
)

model.summary()

## 9. Train

`validation_data` is passed **explicitly**. Do not use `validation_split` — it slices off the tail
of the array, which here would hand you a validation set made of whichever trips happen to sort
last, rather than the trips the `split` column designated.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    shuffle=True,   # fine: each sample is a self-contained window, not a sequence step
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("huber loss"); plt.legend(); plt.title("Training curve")
plt.grid(alpha=0.3)
plt.show()

## 10. Evaluate against the baseline

In [ ]:
y_pred = model.predict(X_test, batch_size=256, verbose=0)

model_mae = report(y_test, y_pred, "LSTM (test)")
print()
baseline_mae = report(y_test, persist_test, "PERSISTENCE (test)")

print("\nimprovement over baseline:")
for i, name in enumerate(TARGETS):
    delta = (baseline_mae[i] - model_mae[i]) / baseline_mae[i] * 100
    verdict = "beats baseline" if delta > 0 else "WORSE THAN BASELINE"
    print(f"  {name:10s} {delta:+6.1f}%   {verdict}")

### Error by forecast horizon

Aggregate MAE hides the shape of the problem. Error should grow with how far ahead you are
predicting; if it is flat, the model is likely just echoing the last input value.

In [ ]:
y_true_inv = inverse_targets(y_test)
y_pred_inv = inverse_targets(y_pred)
persist_inv = inverse_targets(persist_test)

steps = np.arange(1, N_FUTURE + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, (ax, name, unit) in enumerate(zip(axes, TARGETS, ["C", "%RH"])):
    ax.plot(steps, np.abs(y_true_inv[:, :, i] - y_pred_inv[:, :, i]).mean(axis=0), label="LSTM")
    ax.plot(steps, np.abs(y_true_inv[:, :, i] - persist_inv[:, :, i]).mean(axis=0), "--", label="persistence")
    ax.set_xlabel("minutes ahead"); ax.set_ylabel(f"MAE ({unit})")
    ax.set_title(name); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Spot-check a few forecasts

In [ ]:
idx = np.random.choice(len(X_test), 3, replace=False)
fig, axes = plt.subplots(len(idx), 2, figsize=(13, 3.2 * len(idx)))

for r, j in enumerate(idx):
    hist = targ_scaler.inverse_transform(
        X_test[j][:, [FEATURES.index("t_air_c"), FEATURES.index("rh_pct")]]
    )
    for c, (name, unit) in enumerate(zip(TARGETS, ["C", "%RH"])):
        ax = axes[r, c]
        ax.plot(np.arange(-N_PAST, 0), hist[:, c], color="gray", label="history")
        ax.plot(steps, y_true_inv[j, :, c], "o-", ms=3, label="actual")
        ax.plot(steps, y_pred_inv[j, :, c], "s-", ms=3, label="predicted")
        ax.axvline(0, color="k", lw=0.8, ls=":")
        ax.set_title(f"sample {j} - {name}"); ax.set_ylabel(unit); ax.grid(alpha=0.3)
        if r == 0 and c == 0:
            ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 11. Save model and scalers together

The scalers are not optional artifacts — without them the saved model produces numbers in
standardized space that cannot be converted back to °C or %RH.

In [ ]:
import joblib, json, os

os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/coldchain_lstm.keras")
joblib.dump({"feat_scaler": feat_scaler, "targ_scaler": targ_scaler}, "artifacts/scalers.joblib")

with open("artifacts/config.json", "w") as f:
    json.dump({
        "n_past": N_PAST, "n_future": N_FUTURE,
        "features": FEATURES, "cont_feats": CONT_FEATS, "targets": TARGETS,
    }, f, indent=2)

print("saved to artifacts/")

## Where to go if the baseline is hard to beat

`t_air_c` correlates only weakly with every other column in this file (the strongest is
`door_open` at 0.17). Nearly all of the signal lives in the target's own recent history, which
means the persistence baseline is strong and the exogenous sensors may earn less than expected.

Things worth trying, roughly in order of payoff:

1. **Predict the delta instead of the level.** Target `y[t+h] - y[t_last_observed]` and add the
   last value back at inference. This removes the level-matching work and forces the network to
   spend its capacity on the dynamics.
2. **Shorten the horizon.** 30 minutes out is a lot for minute-resolution data. Check the
   per-horizon plot — if the model only wins in the first 5–10 minutes, forecast that instead.
3. **Add engineered lags** — rolling means, and minutes-since-last-door-open / defrost-off. Event
   recency usually carries more signal than the raw binary flag.
4. **Widen `N_PAST`** to 180 or 240 if the compressor duty cycle is longer than 2 hours.
5. **Only then** make the network bigger. Capacity is rarely the binding constraint here.